In [ ]:
from collections import Counter
import screed
import pandas as pd
import os
import re

def reverse_complement(dna_seq):
    """ 
    Return the reverse complement of a DNA sequence. 
    Arguments:
        dna_seq - DNA input sequence as a string
        
    Returns:
        reverse_complement - Reverse complement of DNA input sequence 
    
    """
    complement = {'a': 't', 'A':'T', 't': 'a', 'T':'A','c': 'g','C':'G', 'g': 'c', 'G':'C'}
    
    reverse_complement_seq = ''.join(complement.get(base, base) for base in reversed(dna_seq))
        
    return reverse_complement_seq

def translate(dna_seq):
    """
    Generates a protein sequence from a DNA sequence inputted as a string. It will only convert if the exon is in upper
    case

    Arguments:
        dna_seq(str) - input DNA sequence
        
    Returns:
        protein_sequence(str) - output protein sequence 
    """
    
    #Generate dictionary of codons for dna to protein translation
    
    codon_table = {
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',  # Alanine
    'TGT': 'C', 'TGC': 'C',                          # Cysteine
    'GAT': 'D', 'GAC': 'D',                          # Aspartic acid
    'GAA': 'E', 'GAG': 'E',                          # Glutamic acid
    'TTT': 'F', 'TTC': 'F',                          # Phenylalanine
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',  # Glycine
    'CAT': 'H', 'CAC': 'H',                          # Histidine
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I',              # Isoleucine
    'AAA': 'K', 'AAG': 'K',                          # Lysine
    'TTA': 'L', 'TTG': 'L', 'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',  # Leucine
    'ATG': 'M',                                      # Methionine (Start codon)
    'AAT': 'N', 'AAC': 'N',                          # Asparagine
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',  # Proline
    'CAA': 'Q', 'CAG': 'Q',                          # Glutamine
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R', 'AGA': 'R', 'AGG': 'R',  # Arginine
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S', 'AGT': 'S', 'AGC': 'S',  # Serine
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',  # Threonine
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',  # Valine
    'TGG': 'W',                                      # Tryptophan
    'TAT': 'Y', 'TAC': 'Y',                          # Tyrosine
    'TAA': '*', 'TAG': '*', 'TGA': '*'               # Stop codons
    }
    
    # Initialize the protein sequence
    protein = ""

    # Iterate over the DNA sequence in steps of 3 (codon length)
    for i in range(0, len(dna_seq) - len(dna_seq) % 3, 3):
        
        # Define a codon as 3 bases
        codon = dna_seq[i:i + 3]
        # Translate the codon into an amino acid based on the dictionary
        if codon.isupper():
            protein += codon_table.get(codon, '?')  # '?' is a placeholder for unknown codons
        else:
            protein += 'x'

    return protein

def analyze_fastq(fastq_file, prefix, suffix, wt_amino_acid, variant_codon, variant_amino_acid, sequencing_strand, output_name, integration_specific_primer=None, non_integration_specific_primer=None):
    codon_to_label = {
        'GCC': 'A', 'TGC': 'C', 'GAC': 'D', 'GAG': 'E', 'TTC': 'F', 
        'GGC': 'G', 'CAC': 'H', 'ATC': 'I', 'AAG': 'K', 'CTG': 'L', 
        'ATG': 'M', 'AAC': 'N', 'CCC': 'P', 'CAG': 'Q', 'CGG': 'R', 
        'AGC': 'S', 'ACC': 'T', 'GTG': 'V', 'TGG': 'W', 'TAC': 'Y', 
        'TAA': 'Stop', 
        variant_codon: f'Patient Variant ({variant_codon}/{variant_amino_acid})',    
    }

    # Create regex to capture the 3-base variant
    # This looks for the prefix, captures any 3 bases, and then the suffix
    extract_pattern = re.compile(rf"{prefix}([ATCG]{{3}}){suffix}")
    if integration_specific_primer:
        integration_specific_primer_pattern = re.compile(integration_specific_primer)
    if non_integration_specific_primer:
        non_integration_specific_primer_pattern = re.compile(non_integration_specific_primer)

    counts = Counter()
    total_reads = 0
    all_variants_count = 0
    forward_primer_count = 0
    reverse_primer_count = 0

    for record in screed.open(fastq_file):
        total_reads += 1
        seq = record.sequence
        if integration_specific_primer:
            if integration_specific_primer_pattern.match(seq):
                forward_primer_count += 1
        if non_integration_specific_primer:
            if non_integration_specific_primer_pattern.match(seq):
                reverse_primer_count += 1
        match = extract_pattern.search(seq)
        if match:
            all_variants_count += 1
            codon = match.group(1) # The [ATCG]{3} part
            if sequencing_strand == 'Reverse':
                codon = reverse_complement(codon)
            if codon in codon_to_label:
                label = codon_to_label[codon]
                counts[label] += 1

    final_data = {
        "reads": total_reads,
        "codons": all_variants_count,
        "integration_specific_primer_count": forward_primer_count,
        "non_integration_specific_primer_count": reverse_primer_count,
        }
    for label in codon_to_label.values():
        final_data[label] = counts.get(label, 0)


    df = pd.DataFrame(list(final_data.items()), columns=["Label", "Counts"])
    df.loc[df['Label'] == wt_amino_acid, 'Label'] = f'{wt_amino_acid} (WT)'
    df['Name'] = output_name # Add a 'Name' column that's the same as 'Label'
    display(df)
    df.to_excel (output_name + '.xlsx', index=False)
    
    return 

In [ ]:
#FOR BULK ANALYSIS OF Innoculum FASTQ FILES
fastq_analysis_inputs_dir = "fastq_inputs/"
fastq_file_path = r"<path/fastq.gz"
output_folder = "outputs/"

for file in os.listdir(fastq_analysis_inputs_dir):
        name = os.path.splitext(file)[0]  # Get filename without extension
        print(f"Processing {name}")
        df = pd.read_excel(f'{fastq_analysis_inputs_dir}{name}.xlsx')
        display(df)
        sequencing_strand = df['sequencing_strand'][0]
        prefix = df['inn_prefix'][0].upper()
        suffix = df['inn_suffix'][0].upper()
        wt_amino_acid = df['wt_amino_acid'][0].upper()
        variant_codon = df['variant_codon'][0].upper()
        variant_amino_acid = df['variant_amino_acid'][0].upper()
        integration_specific_primer = df['integration_specific_primer_for_off_target_checks'][0].upper()
        non_integration_specific_primer = df['non_integration_specific_primer_for_off_target_checks'][0].upper()
        analyze_fastq(fastq_file = fastq_file_path, prefix = prefix, suffix = suffix, wt_amino_acid = wt_amino_acid, variant_codon = variant_codon, variant_amino_acid = variant_amino_acid, sequencing_strand = sequencing_strand, output_name = f'{output_folder}{name}', integration_specific_primer = integration_specific_primer, non_integration_specific_primer = non_integration_specific_primer)
          

In [ ]:
#FOR BULK ANALYSIS OF Integration FASTQ FILES
fastq_analysis_inputs_dir = "fastq_inputs/"
fastq_file_path = r"<path/fastq.gz"
output_folder = "outputs/"

for file in os.listdir(fastq_analysis_inputs_dir):
        name = os.path.splitext(file)[0]  # Get filename without extension
        print(f"Processing {name}")
        df = pd.read_excel(f'{fastq_analysis_inputs_dir}{name}.xlsx')
        display(df)
        sequencing_strand = df['sequencing_strand'][0]
        prefix = df['isp_prefix'][0].upper()
        suffix = df['isp_suffix'][0].upper()
        wt_amino_acid = df['wt_amino_acid'][0].upper()
        variant_codon = df['variant_codon'][0].upper()
        variant_amino_acid = df['variant_amino_acid'][0].upper()
        integration_specific_primer = df['integration_specific_primer_for_off_target_checks'][0].upper()
        non_integration_specific_primer = df['non_integration_specific_primer_for_off_target_checks'][0].upper()
        analyze_fastq(fastq_file = fastq_file_path, prefix = prefix, suffix = suffix, wt_amino_acid = wt_amino_acid, variant_codon = variant_codon, variant_amino_acid = variant_amino_acid, sequencing_strand = sequencing_strand, output_name = f'{output_folder}{name}', integration_specific_primer = integration_specific_primer, non_integration_specific_primer = non_integration_specific_primer)  